# Step 10: Evaluate the Root-Aligned SMILES checkpoint on ORD + USPTO (Colab GPU)

Kaggle training finished (`kaggle/04_train_reactant_rootaligned.ipynb`): 3 epochs, `train_runtime=5477s` (~91min),
matches the proven DDP rate. This notebook evaluates that checkpoint on the standard 300-record
ORD and USPTO eval sets (same ones variant 2/4/5 were scored on in `RESULTS.md`), so the result is
directly comparable.

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU
accelerator** (T4).

**Checkpoint upload required:** local folder
`/private/tmp/claude-502/-Users-Oleh-Documents-diploma-ldit/c4eab724-2366-4d6f-85ee-7f44c967845d/scratchpad/kaggle_output_rootaligned/model1_reactant_rootaligned57k/final`
(~758MB) -- upload it to Google Drive (drag-and-drop on drive.google.com is more reliable than the
browser file picker for large folders), then point `checkpoint_path` below at wherever it lands.

Uses `--batch-size 32` for real GPU utilization (batch=1 leaves a T4 mostly idle -- see
`run_reactiont5_topk.py`'s own `--batch-size` help text). The script also auto-fixes a legacy
`tokenizer_config.json` format issue on load (`extra_special_tokens` list->dict) that affects every
checkpoint fine-tuned earlier in this project -- no manual step needed, `git pull`/clone already
gets the fix.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner
!git pull

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
checkpoint_path = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_rootaligned57k/final"  # @param {type:"string"}

import os
assert os.path.isdir(checkpoint_path), f"Not found: {checkpoint_path} -- did you upload it to Drive and set the path above?"
print("Checkpoint found:", checkpoint_path)

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model "{checkpoint_path}" \
    --num-beams 10 --device cuda --batch-size 32 \
    --output experiments/v2_model1_topk/rootaligned57k_ord_topk.json

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_uspto_eval_targets.json \
    --t5-model "{checkpoint_path}" \
    --num-beams 10 --device cuda --batch-size 32 \
    --output experiments/v2_model1_topk/rootaligned57k_uspto_topk.json

**When both finish:** each cell prints its own summary at the end. Download both
`experiments/v2_model1_topk/rootaligned57k_*_topk.json` files (Colab file browser, left sidebar) and
bring them back to the local repo's `experiments/v2_model1_topk/` so RESULTS.md can be updated the
same way as every other variant.